# BCT Hackathon User Modelling 
> Version : 2 
## Goal 
Build an agent that understands users deeply enough to simulate their reviews — capturing tone, rating behaviour, and contextual nuance.
- Simulate star ratings and written reviews for 
unseen items
- Leverage user history, item metadata, and 
contextual signals
- Evaluated on review quality, rating accuracy, 
and behavioural fidelity

### Notebook version 2 
This is the final version of the first build of the it entails downloading the dataset from hugging face (we used the beauty dataset first )
- then perfroming some much needed Exploratory Data Analysis on the data.
- then we created a function to clean the data into review rich data >10 or 100 words reviews
- then we used a hardcoded-review persona builder it builds persona on the reviewer using thier reviews
- then finally it uses gemini 2.5 to generate reviews
- then we then evaluate the data

## Notebook version 3 
- check the evaluation data (hope its not leaking we should be spliting the reviews per users )
- use a multi-agent workflow one agent gets the persona another builds the reviews
- try techniques to save tokens 
- try cross-domain reviewing 

In [1]:
# ── INSTALL FIRST ────────────────────────────────────────────────
!pip install datasets

## importing dependencies

In [2]:
import json 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os

# ── DOWNLOAD ─────────────────────────────────────────────────────
!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz" \
    -O "/kaggle/working/All_Beauty_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz" \
    -O "/kaggle/working/meta_All_Beauty.jsonl.gz"

# confirm sizes — should be several MB each
!ls -lh /kaggle/working/*.gz

/kaggle/working/All 100%[===================>]  90.07M  3.45MB/s    in 58s     
/kaggle/working/met 100%[===================>]  38.02M  4.64MB/s    in 9.6s    
-rw-r--r-- 1 root root 91M Jan 16  2025 /kaggle/working/All_Beauty_reviews.jsonl.gz
-rw-r--r-- 1 root root 39M Jan 16  2025 /kaggle/working/meta_All_Beauty.jsonl.gz


In [4]:
import pandas as pd

# ── LOAD ─────────────────────────────────────────────────────────
print("Loading reviews...")
reviews_df = pd.read_json(
    '/kaggle/working/All_Beauty_reviews.jsonl.gz',
    lines       = True,
    compression = 'gzip'
)
print(f"Reviews : {reviews_df.shape}")
print(f"Columns : {reviews_df.columns.tolist()}")

print("\nLoading metadata...")
meta_df = pd.read_json(
    '/kaggle/working/meta_All_Beauty.jsonl.gz',
    lines       = True,
    compression = 'gzip'
)
print(f"Metadata : {meta_df.shape}")
print(f"Columns  : {meta_df.columns.tolist()}")

Loading reviews...
Reviews : (701528, 10)
Columns : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Loading metadata...
Metadata : (112590, 14)
Columns  : ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']


In [5]:
# ── JOIN ─────────────────────────────────────────────────────────
# flatten list fields in metadata
meta_df['description_text'] = meta_df['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) and x else ''
)
meta_df['features_text'] = meta_df['features'].apply(
    lambda x: ' | '.join(x[:3]) if isinstance(x, list) and x else ''
)

# slim metadata down
meta_slim = meta_df[[
    'parent_asin',
    'title',
    'description_text',
    'features_text',
    'price',
    'store',
    'main_category'
]].drop_duplicates(subset='parent_asin')

# join
df = reviews_df.merge(meta_slim, on='parent_asin', how='left')

# rename to match your pipeline
df = df.rename(columns={

    'title_x'          : 'review_title',
    'title_y'          : 'product_title',
})

print(f"\nJoined shape : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")


Joined shape : (701528, 16)
Columns      : ['rating', 'review_title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description_text', 'features_text', 'price', 'store', 'main_category']


In [6]:
# ── HEALTH CHECK ─────────────────────────────────────────────────
user_counts = df.groupby('user_id').size()
viable      = user_counts[user_counts >= 20]

print("=" * 50)
print("HEALTH CHECK")
print("=" * 50)
print(f"Total reviews         : {len(df):,}")
print(f"Unique users          : {df['user_id'].nunique():,}")
print(f"Unique products       : {df['asin'].nunique():,}")
print(f"Users with 20+ reviews: {len(viable):,}")
print(f"Avg review length     : {df['text'].str.split().str.len().mean():.0f} words")
print(f"Rating distribution   : {df['rating'].value_counts().sort_index().to_dict()}")
#print(f"Missing product title : {df['product_title'].isna().sum():,}")
#print(f"Missing description   : {df['description_text'].isna().sum():,}")
print(f"Verified purchase %   : {df['verified_purchase'].mean()*100:.1f}%")

HEALTH CHECK
Total reviews         : 701,528
Unique users          : 631,986
Unique products       : 115,709
Users with 20+ reviews: 117
Avg review length     : 33 words
Rating distribution   : {1: 102080, 2: 43034, 3: 56307, 4: 79381, 5: 420726}
Verified purchase %   : 90.5%


In [7]:
df.head()

,rating,review_title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,Herbivore - Natural Sea Mist Texturizing Salt ...,"If given the choice, weÕd leave most telltale ...",,NaN,HERBIVORE,All Beauty
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,All Natural Vegan Dry Shampoo Powder - Eco Fri...,,,NaN,Two Goats Apothecary,All Beauty
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,New Road Beauty - Creamsicle - Variety 3 Pack ...,New Road Beauty Paraffin Wax is recommended fo...,"Same Great Product, NEW PACKAGING. | MOISTURIZ...",21.98,New Road Beauty,All Beauty
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,muaowig Ombre Body Wave Bundles 1B Grey Human ...,Hair Material: Brazilian Virgin Human Hair Bun...,?Hair Bundle Material?:Brazilian Virgin Human ...,NaN,muaowig,All Beauty
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True,Yinhua Electric Nail Drill Kit Portable Profes...,,,NaN,Yinhua,All Beauty


## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [8]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
duplicate_df = df.drop("images",axis = 1)
print(f"\nDuplicate rows: {duplicate_df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (701528, 16)

Column dtypes:
rating                        int64
review_title                 object
text                         object
images                       object
asin                         object
parent_asin                  object
user_id                      object
timestamp            datetime64[ns]
helpful_vote                  int64
verified_purchase              bool
product_title                object
description_text             object
features_text                object
price                       float64
store                        object
main_category                object
dtype: object

Missing values:
rating                    0
review_title              0
text                      0
images                    0
asin                      0
parent_asin               0
user_id                   0
timestamp                 0
helpful_vote              0
verified_purchase         0
product_title             0
description_text        

### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [9]:
## understanding the schema of the review dataset 
df.iloc[0].to_dict()

{'rating': 5,
 'review_title': 'Such a lovely scent but not overpowering.',
 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
 'images': [],
 'asin': 'B00YQ6X8EO',
 'parent_asin': 'B00YQ6X8EO',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': Timestamp('2020-05-05 14:08:48.923000'),
 'helpful_vote': 0,
 'verified_purchase': True,
 'product_title': 'Herbivore - Natural Sea Mist Texturizing Salt Spray (Coconut, 8 oz)',
 'description_text': 'If given the choice, weÕd leave most telltale signs of the beachÑsunburns, sandy toes, crab claw pinches, etc.Ñat the beach where they belong. The one thing wish we could take with us? That salty sea breeze. This all-natural spray manages, magically, to bottle the effect of sea mist s

In [10]:
#checking the columns in the data set 
df.columns.tolist()

['rating',
 'review_title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'product_title',
 'description_text',
 'features_text',
 'price',
 'store',
 'main_category']

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona

- the results here are quite worse to build the prototype i will use users with 5 or more reviews 

In [11]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

count    631986.000000
mean          1.110037
std           0.753202
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         165.000000
dtype: float64

In [12]:
# How many viable user?
five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")


The number of users with five reviews or more are 1620,
        The number of users with ten reviews or more are 330,
        while the number of users with twenty reviews or more are 117


In [13]:
# Distribution shape
user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

1     583553
2      39274
3       5713
4       1826
5        558
6        351
7        181
8        130
9         70
11        35
10        35
13        33
12        26
16        20
15        20
14        19
17        14
21         9
26         8
24         8
Name: count, dtype: int64

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [14]:
# Global rating distribution
df['rating'].value_counts().sort_index()

rating
1    102080
2     43034
3     56307
4     79381
5    420726
Name: count, dtype: int64

In [15]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

count    631986.000000
mean          3.948681
std           1.487903
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [16]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

count    48433.000000
mean         0.697705
std          0.918379
min          0.000000
25%          0.000000
50%          0.000000
75%          1.414214
max          2.828427
Name: rating, dtype: float64

In [17]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

count    1.620000e+03
mean    -1.484943e-02
std      2.539332e-01
min     -1.200000e+00
25%     -8.571429e-02
50%      2.013140e-16
75%      5.714286e-02
max      1.100000e+00
Name: rating_slope, dtype: float64
trend
insufficient_data        630366
consistent                  732
increasingly_critical       477
increasingly_generous       411
Name: count, dtype: int64


In [18]:
df.head()

,rating,review_title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq
72840,5,Five Stars,Great product....excellent price for good resu...,[],B013HR1A92,B013HR1A92,AE222BBOVZIF42YOOPNBXL4UUMYA,2016-03-10 00:27:52.000,0,True,UNGLINGA Black Mask Blackhead Remover Purifyin...,"CONCERNS: Enlarged pore, Blackheads, Anti-agin...",DEEP CLEANSING BLACKHEAD REMOVER WITH TOOL - O...,NaN,UNGLINGA,All Beauty,1
392969,5,Nice consistency and great smell,[[VIDEOID:8b3489ae8e6301ceb95a2973d7f721f3]],[],B0BTT658PQ,B0BTT658PQ,AE222FP7YRNFCEQ2W3ZDIGMSYTLQ,2023-03-07 03:10:12.143,0,True,Hair Growth Serum for Women and Men – 100% Mad...,,,NaN,Svvimer,All Beauty,1
636591,5,Wow,It tastes good,[],B00PBDMRES,B00PBDMRES,AE222X475JC6ONXMIKZDFGQ7IAUA,2017-01-06 18:55:24.000,2,True,1775 Valor For Men 3.4 oz EDT Spray By Royal C...,"The Modern Maverick""The true essence of streng...",,NaN,1775 VALOR,All Beauty,1
293687,4,Lensoclean Unit,The cleaning unit does a good job of cleaning ...,[],B00012FPSO,B00012FPSO,AE222Y4WTST6BUZ4J5Y2H6QMBITQ,2013-06-24 21:11:42.000,1,True,drtulz ULTRASONIC CLEANING,The product has EUROPLUG. For US market needs ...,1111,NaN,drtulz,All Beauty,1
589847,5,Sus colores,Son como en la foto,[],B07QNPXBLH,B07QNPXBLH,AE2232TEZOEWQLAFEX2NA6VBGMYQ,2019-07-30 05:47:26.197,0,True,SIQUK 12 Pieces Headbands with Twist Knot Head...,,,NaN,SIQUK,All Beauty,1


In [19]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

"                 timestamp  rating  review_seq                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             text\n242833 2016-07-04 20:3

In [20]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n📊 QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f} ⭐  (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [21]:

"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

USER PROFILE: AHBWH2LBU3NFLD46GKJKIBAHKXEQ

📊 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.05 ⭐  (std: 1.00)
   Avg review length: 104 words
   Rating breakdown: {1: 1, 2: 3, 3: 3, 4: 18, 5: 14}
   Rating trend    : ↑ more generous over time  (early avg: 3.89 → late avg: 4.20)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Aug 2020  |  ⭐⭐⭐⭐ (4/5)  |  237 words
     I try to avoid using traditional files and emery boards
     on my nails. I take a lot of medications that make my
     nails weak and brittle, and any surface that's too
     abrasive wreaks havoc on my fingertips. I can't carry a
     full size glass nail file with me everywhere, so I was
     on the hunt for a travel sized file that could fit in
     my pocket, wallet, or purse. When these came up, I
     thought I'd give them a shot. The pros are the files
     are a great size, they're packaged well, and I see them
     lasting for awhile. 

In [22]:
# random user with 30+ reviews
user_history = profile_user(df)

Randomly selected user: AHPGHDFIU3BUB3RQBP56RQQA7W4Q

USER PROFILE: AHPGHDFIU3BUB3RQBP56RQQA7W4Q

📊 QUICK STATS
   Total reviews   : 57
   Avg rating      : 4.61 ⭐  (std: 0.56)
   Avg review length: 250 words
   Rating breakdown: {3: 2, 4: 18, 5: 37}
   Rating trend    : ↑ more generous over time  (early avg: 4.39 → late avg: 4.83)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Mar 2020  |  ⭐⭐⭐ (3/5)  |  142 words
     Due to the costs, one cannot expect to get a high
     quality human hair wig. Rather, with the proper
     expectations, this is a decent, synthetic strand, wig.
     In fact, the strands look like very fine, real hair. I
     would say that it is very similar to Chinese hair,
     which is fine, and very straight.<br />Being 14" long,
     you have the choice of leaving it at the current
     shoulder length, or styling and cutting it. I like that
     aspect, making it more versatile.  This niche produ

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

In [23]:
# Review length distribution
df['text'].str.split().str.len().describe()

count    701528.000000
mean         32.750720
std          45.973273
min           0.000000
25%           8.000000
50%          19.000000
75%          40.000000
max        2585.000000
Name: text, dtype: float64

In [24]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

74841

In [25]:
# Avg text length per user
avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [26]:
display(avg_text_per_user)

user_id
AE222BBOVZIF42YOOPNBXL4UUMYA     52.0
AE222FP7YRNFCEQ2W3ZDIGMSYTLQ     44.0
AE222X475JC6ONXMIKZDFGQ7IAUA     14.0
AE222Y4WTST6BUZ4J5Y2H6QMBITQ    184.0
AE2232TEZOEWQLAFEX2NA6VBGMYQ     19.0
                                ...  
AHZZYVEU6QFMPFZ2HJUWR22SNK4A     11.0
AHZZZAK24AJ3JNBDUZJGHHWSRVAA    226.0
AHZZZJP24QUSB5XWW6MAXYBZZZSQ     33.0
AHZZZL7YQJA3RSA6PYK3WMFACYIQ    114.0
AHZZZSOTVOVACVK2WWXL4ITEAPIA     17.0
Name: text, Length: 631986, dtype: float64

In [27]:
#print(avg_text_per_user.value_counts())


In [28]:
# Verified purchase flag

df['verified_purchase'].value_counts()
# prefer verified = True

verified_purchase
True     634969
False     66559
Name: count, dtype: int64

## Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data 

In [33]:
import pandas as pd

def clean_amazon_reviews(df, 
                          min_reviews=20, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop("images", axis = 1)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining               : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [34]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

AMAZON REVIEWS — CLEANING PIPELINE

▶ Starting shape: 701,528 rows × 17 columns

[1] Duplicate rows removed   : 0
    Remaining                : 701,528

[2] Duplicate columns removed: 0
    Remaining columns        : ['rating', 'review_title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description_text', 'features_text', 'price', 'store', 'main_category', 'review_seq']

[3] Rows dropped (missing critical fields): 0
    Remaining                              : 701,528

[4] Rows dropped (empty / short reviews) : 456,668
    Min word count threshold             : 30 words
    Remaining                            : 244,860

[5] Rows dropped (unverified purchases)  : 41,244
    Remaining                            : 203,616

[6] Rows dropped (invalid ratings)       : 0
    Remaining                            : 203,616

[7] Users dropped (< 10 reviews)          : 192,293
    Viable users remaining               : 7
    Rows

In [35]:
rich_df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq,word_count
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,"Flawless Finish Foundation, Colour Changing Fo...",,,NaN,CIDBEST,All Beauty,1,158
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,"Flawless Liquid Foundation Cream, Liquid Found...",,,NaN,Cherioll,All Beauty,2,97
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,"Rapid Reduction Eye Cream, Under Eye Cream, Un...",,,NaN,CIDBEST,All Beauty,3,137
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,"Concealer Cream,Makeup concealer,Skin Lighteni...",,,NaN,SCOBUTY,All Beauty,4,132
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,W-Airfit Primer Face Makeup Base Pink Isolatio...,,,NaN,Lofu,All Beauty,5,124


In [37]:
rich_df.to_csv('rich_users.csv', index=False)

# Part 2 Build a Persona 

## Persona style 

In [38]:

import re

# ── STEP 1: STATISTICAL PERSONA (no LLM needed) ─────────────────

def extract_statistical_persona(df):
    """
    Extracts measurable traits per user from their review history.
    No LLM call yet — pure signal extraction from data.
    """

    def rating_trend(ratings):
        """slope of ratings over time — positive = more generous, negative = more critical"""
        if len(ratings) < 4:
            return 0.0
        x = np.arange(len(ratings))
        slope = np.polyfit(x, ratings, 1)[0]
        return round(float(slope), 4)

    def writing_style(texts):
        """extract fingerprint signals from review text"""
        texts = [str(t) for t in texts if len(str(t)) > 10]
        if not texts:
            return {}
        
        all_sentences = []
        all_words     = []
        openers       = []
        
        for text in texts:
            sentences = re.split(r'[.!?]+', text)
            sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
            words     = text.lower().split()
            
            all_sentences.extend(sentences)
            all_words.extend(words)
            if words:
                openers.append(words[0])  # first word of each review
        
        avg_sentence_len = np.mean([len(s.split()) for s in all_sentences]) if all_sentences else 0
        vocab_richness   = len(set(all_words)) / len(all_words) if all_words else 0  # type-token ratio
        avg_review_len   = np.mean([len(t.split()) for t in texts])
        
        # is this person verbose or terse?
        verbosity = "verbose" if avg_review_len > 80 \
                    else "moderate" if avg_review_len > 40 \
                    else "terse"
        
        # most common opener words (excluding stopwords)
        stopwords = {'i', 'the', 'this', 'a', 'an', 'it', 'my', 'we', 'so'}
        meaningful_openers = [w for w in openers if w not in stopwords]
        top_opener = meaningful_openers[0] if meaningful_openers else "unknown"
        
        return {
            "avg_sentence_len"  : round(avg_sentence_len, 1),
            "vocab_richness"    : round(vocab_richness, 3),
            "avg_review_len"    : round(avg_review_len, 1),
            "verbosity"         : verbosity,
            "common_opener"     : top_opener
        }

    def complaint_signals(texts):
        """detect recurring complaint or praise patterns"""
        all_text = " ".join([str(t).lower() for t in texts])
        
        signals = {
            "mentions_price"    : bool(re.search(r'\b(price|expensive|cheap|cost|worth|value|money)\b', all_text)),
            "mentions_quality"  : bool(re.search(r'\b(quality|durable|broke|lasted|cheap|flimsy|solid)\b', all_text)),
            "mentions_shipping" : bool(re.search(r'\b(shipping|delivery|arrived|package|days|late|fast)\b', all_text)),
            "mentions_service"  : bool(re.search(r'\b(service|support|help|staff|seller|response)\b', all_text)),
            "uses_caps"         : bool(re.search(r'\b[A-Z]{3,}\b', " ".join([str(t) for t in texts]))),
            "uses_exclamation"  : " ".join([str(t) for t in texts]).count("!") > len(texts)
        }
        return signals

    def rating_profile(ratings):
        """classify this user's rating personality"""
        avg = np.mean(ratings)
        std = np.std(ratings)
        
        if avg >= 4.2:
            generosity = "generous"
        elif avg <= 2.8:
            generosity = "harsh"
        else:
            generosity = "balanced"
        
        consistency = "consistent" if std < 0.8 else "variable"
        
        return {
            "generosity"   : generosity,
            "consistency"  : consistency,
            "never_gives_5": int(5 not in ratings),
            "never_gives_1": int(1 not in ratings)
        }

    # ── AGGREGATE PER USER ───────────────────────────────────────
    rows = []

    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp') if 'timestamp' in group.columns else group

        ratings  = group['rating'].tolist()
        texts    = group['text'].tolist()

        style    = writing_style(texts)
        signals  = complaint_signals(texts)
        profile  = rating_profile(ratings)

        row = {
            "user_id"      : user_id,

            # --- rating stats ---
            "review_count"    : len(ratings),
            "avg_rating"      : round(np.mean(ratings), 2),
            "rating_std"      : round(np.std(ratings), 2),
            "rating_trend"    : rating_trend(ratings),

            # --- rating profile ---
            "generosity"      : profile["generosity"],
            "consistency"     : profile["consistency"],
            "never_gives_5"   : profile["never_gives_5"],
            "never_gives_1"   : profile["never_gives_1"],

            # --- writing style ---
            "verbosity"       : style.get("verbosity"),
            "avg_review_len"  : style.get("avg_review_len"),
            "vocab_richness"  : style.get("vocab_richness"),
            "avg_sentence_len": style.get("avg_sentence_len"),
            "common_opener"   : style.get("common_opener"),

            # --- complaint / praise signals ---
            "mentions_price"  : signals["mentions_price"],
            "mentions_quality": signals["mentions_quality"],
            "mentions_shipping": signals["mentions_shipping"],
            "mentions_service": signals["mentions_service"],
            "uses_caps"       : signals["uses_caps"],
            "uses_exclamation": signals["uses_exclamation"],

            # --- sample reviews for LLM trait extraction (next step) ---
            "sample_reviews"  : texts[:5],
            "reviewed_items"  : group['asin'].tolist() if 'asin' in group.columns else []
        }
        rows.append(row)

    persona_df = pd.DataFrame(rows)
    print(f"✅ Statistical personas extracted for {len(persona_df):,} users")
    print(f"   Columns: {list(persona_df.columns)}")
    return persona_df


# ── RUN IT ───────────────────────────────────────────────────────
persona_df_claude = extract_statistical_persona(rich_df)

✅ Statistical personas extracted for 7 users
   Columns: ['user_id', 'review_count', 'avg_rating', 'rating_std', 'rating_trend', 'generosity', 'consistency', 'never_gives_5', 'never_gives_1', 'verbosity', 'avg_review_len', 'vocab_richness', 'avg_sentence_len', 'common_opener', 'mentions_price', 'mentions_quality', 'mentions_shipping', 'mentions_service', 'uses_caps', 'uses_exclamation', 'sample_reviews', 'reviewed_items']


In [39]:
persona_df_claude.head()

,user_id,review_count,avg_rating,rating_std,rating_trend,generosity,consistency,never_gives_5,never_gives_1,verbosity,...,avg_sentence_len,common_opener,mentions_price,mentions_quality,mentions_shipping,mentions_service,uses_caps,uses_exclamation,sample_reviews,reviewed_items
0,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,11,4.64,0.48,0.0455,generous,consistent,0,1,verbose,...,16.1,these,True,True,True,True,True,False,[This makeup is crazy. It comes out of the bo...,"[B07VNQ4G13, B07VML1QZC, B07X1PH59J, B07XNYVBY..."
1,AEYKTZXAWOPJG5MGGMKBLRJR6Q3A_2,21,4.95,0.21,0.0052,generous,consistent,0,1,moderate,...,41.9,impressive,True,True,True,False,False,False,[Impressive device. Operates on by charge on u...,"[B07WMFRDZ3, B07QYD1J5C, B07XRJC4TY, B07YSC7V2..."
2,AFR4BTNWATNG7O4SOME4XK6NNBYQ,10,4.80,0.40,0.0848,generous,consistent,0,1,moderate,...,11.8,what,False,False,True,False,True,False,[The bar is rather small..but the jasmine is o...,"[B0013EHI6G, B00CBY2W4U, B004FO8KPE, B00TJE4JU..."
3,AFSCJNRG4BAGAB37REW4XCDPE6XA,11,5.00,0.00,0.0000,generous,consistent,0,1,moderate,...,10.7,tired,True,False,False,False,False,False,[My first experience with a shaver Like this a...,"[B07RG332M3, B07QQQXZ4S, B07Q3BSDC8, B07V6BK6H..."
4,AFUBMCVI5J6G4F2RFQGTDRXLE6VQ,10,3.40,1.11,-0.0364,balanced,variable,0,1,verbose,...,11.2,fantastic,True,False,True,True,True,False,[fantastic value for fake hair. perfect color...,"[B00D93WF26, B00IJVVBG4, B089XX8C33, B08D3HBC3..."


## Prompt Building

In [40]:
def build_user_prompt(user_row, item_asin, item_title, item_description, nigerian_mode=False ):
    """
    Builds a persona-conditioned prompt for review simulation.
    Designed for Gemini API. Uses chain-of-thought before generation.
    
    Parameters:
        user_row         : one row from persona_df
        item_asin        : product ID
        item_title       : product name
        item_description : product description
        nigerian_mode    : inject Nigerian cultural conditioning layer
    """

    # ── 1. BUILD TRAIT SUMMARY ───────────────────────────────────
    # translate extracted traits into natural language descriptions
    # so the LLM gets character, not just numbers

    trait_lines = []

    # rating personality
    generosity   = user_row.get('generosity', 'balanced')
    consistency  = user_row.get('consistency', 'consistent')
    avg_rating   = user_row.get('avg_rating', 3.0)
    rating_std   = user_row.get('rating_std', 1.0)
    rating_trend = user_row.get('rating_trend', 0.0)

    trait_lines.append(f"- They are a {generosity} rater — their average rating is {avg_rating:.1f}/5")
    trait_lines.append(f"- They are {consistency} in their ratings (std dev: {rating_std:.2f})")

    if user_row.get('never_gives_5'):
        trait_lines.append("- They NEVER give 5 stars — even for things they like")
    if user_row.get('never_gives_1'):
        trait_lines.append("- They have never given 1 star — even when disappointed")
    if rating_trend > 0.05:
        trait_lines.append("- Their ratings have been trending upward — they are becoming more generous over time")
    elif rating_trend < -0.05:
        trait_lines.append("- Their ratings have been trending downward — they are becoming more critical over time")

    # writing style
    verbosity        = user_row.get('verbosity', 'moderate')
    avg_review_len   = int(user_row.get('avg_review_len', 50))
    vocab_richness   = user_row.get('vocab_richness', 0.5)
    avg_sentence_len = user_row.get('avg_sentence_len', 15)
    common_opener    = user_row.get('common_opener', '')

    trait_lines.append(f"- Writing style: {verbosity} — they write around {avg_review_len} words per review")
    trait_lines.append(f"- Sentence length: avg {avg_sentence_len:.0f} words per sentence")
    if vocab_richness > 0.7:
        trait_lines.append("- They use rich, varied vocabulary — rarely repeat the same words")
    elif vocab_richness < 0.4:
        trait_lines.append("- They use simple, repetitive vocabulary — plain everyday language")
    if common_opener:
        trait_lines.append(f"- They often start their reviews with words like: '{common_opener}'")

    # complaint / praise signals
    if user_row.get('mentions_price'):
        trait_lines.append("- Price and value for money is a recurring theme in their reviews")
    if user_row.get('mentions_quality'):
        trait_lines.append("- Product quality and durability is something they always comment on")
    if user_row.get('mentions_shipping'):
        trait_lines.append("- They frequently mention shipping speed and delivery experience")
    if user_row.get('mentions_service'):
        trait_lines.append("- Customer service experience often appears in their reviews")
    if user_row.get('uses_exclamation'):
        trait_lines.append("- They use exclamation marks frequently — expressive and emotional tone")
    if user_row.get('uses_caps'):
        trait_lines.append("- They occasionally use ALL CAPS for emphasis")

    traits_block = "\n".join(trait_lines)

    # ── 2. FORMAT SAMPLE REVIEWS ─────────────────────────────────
    samples = user_row.get('sample_reviews', [])
    if samples:
        sample_block = "\n\n".join([
            f"  Past review {i+1}:\n  \"{str(r).strip()}\""
            for i, r in enumerate(samples[:4])  # max 4 samples
        ])
    else:
        sample_block = "  No sample reviews available."

    # ── 3. NIGERIAN CONDITIONING LAYER ───────────────────────────
    nigerian_block = ""
    if nigerian_mode:
        nigerian_block = """
IMPORTANT CULTURAL CONTEXT:
This reviewer is Nigerian. Their review should naturally reflect Nigerian consumer behaviour:
- They are price-conscious and often compare value to local alternatives
- They may reference familiar Nigerian brands, experiences, or comparisons ("like the ones in Shoprite", "better than what you find for Ikeja")
- They may code-switch lightly — mixing standard English with Pidgin phrases naturally (e.g. "e good sha", "I no go lie", "the thing dey work")
- Do NOT force Pidgin on every sentence — use it sparingly and naturally, the way an educated Nigerian would
- They may mention community or family context ("my whole family", "my colleagues at work")
- Scepticism about product claims is common — they may note what surprised them vs expectation
"""

    # ── 4. RATING ANCHOR ─────────────────────────────────────────
    # give the LLM a statistical prior so rating prediction stays
    # close to this user's actual behaviour — reduces RMSE significantly
    low  = max(1, round(avg_rating - rating_std))
    high = min(5, round(avg_rating + rating_std))
    rating_anchor = f"Based on their history, their rating will most likely fall between {low} and {high} stars."

    # ── 5. ASSEMBLE THE FULL PROMPT ──────────────────────────────
    prompt = f"""You are simulating the review behaviour of a specific, real Amazon user.
Your job is NOT to write a good review — your job is to write the review THIS SPECIFIC PERSON would write.
Stay in character throughout. Do not default to generic reviewer behaviour.

═══════════════════════════════════════════
WHO THIS PERSON IS
═══════════════════════════════════════════
{traits_block}

═══════════════════════════════════════════
SAMPLES OF HOW THEY ACTUALLY WRITE
═══════════════════════════════════════════
{sample_block}
{nigerian_block}
═══════════════════════════════════════════
THE PRODUCT THEY ARE REVIEWING (UNSEEN)
═══════════════════════════════════════════
Product title      : {item_title}
Product ID (ASIN)  : {item_asin}
Product description: {item_description}

═══════════════════════════════════════════
INSTRUCTIONS
═══════════════════════════════════════════
Before writing the review, reason briefly:
- What would this person notice about this product given their traits?
- What would they likely complain about or praise?
- What rating does their history suggest they would give?

{rating_anchor}

Then produce your output in EXACTLY this format — nothing before, nothing after:

REASONING: [2-3 sentences of internal reasoning about this user and this product]
RATING: [single integer 1-5]
TITLE: [short review headline in their voice]
REVIEW: [full review text in their voice and style]
""".strip()

    return prompt

In [41]:
# use the kaggle secret enviroment to secure my api key 
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

In [42]:
import google.generativeai as genai

genai.configure(api_key= GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

def simulate_review(user_row, item_asin, item_title, item_description, nigerian_mode=False):
    """
    Full pipeline: build prompt → call Gemini → parse output
    """
    prompt = build_user_prompt(
        user_row, item_asin, item_title, 
        item_description, nigerian_mode
    )
    
    response = model.generate_content(prompt)
    raw      = response.text.strip()
    
    # ── PARSE THE STRUCTURED OUTPUT ──────────────────────────
    result = {
        "user_id"   : user_row['user_id'],
        "asin"      : item_asin,
        "raw_output": raw
    }
    
    for field in ['REASONING', 'RATING', 'TITLE', 'REVIEW']:
        pattern = rf"{field}:\s*(.*?)(?=\n[A-Z]+:|$)"
        match   = re.search(pattern, raw, re.DOTALL)
        result[field.lower()] = match.group(1).strip() if match else None
    
    # cast rating to int safely
    try:
        result['rating'] = int(result['rating'])
    except (TypeError, ValueError):
        result['rating'] = round(user_row['avg_rating'])  # fallback to user avg
    
    return result

In [44]:
# grab one user from persona_df
test_user = persona_df_claude.iloc[0]

result = simulate_review(
    user_row         = test_user,
    item_asin        = "B09XYZ123",
    item_title       = "Beauty Cream",
    item_description = "Beauty Cream for soft skin 24hrs",
    nigerian_mode    = False
)

print("REASONING :", result['reasoning'])
print("RATING    :", result['rating'])
print("TITLE     :", result['title'])
print("REVIEW    :", result['review'])

REASONING : This user is a generous rater who focuses on immediate product effects, durability, value for money, and delivery experience. For a "Beauty Cream for soft skin 24hrs", they would specifically comment on how fast it makes skin soft, if the softness lasts for 24 hours, and how it feels during application. Given the product's basic description and their positive rating history, a highly positive review with emphasis on effectiveness and good value is expected.
RATING    : 5
TITLE     : This Cream Is So Good
REVIEW    : This beauty cream is really good. It makes your skin feel soft right away. It goes on smooth, just like a regular moisturizer. It says it's for soft skin for 24 hours, and it really does a good job keeping my skin soft for a long, long time. It doesn't feel greasy or weird after applying. It just soaks in nicely and leaves your skin feeling smooth all day. I have used other creams that don't last as long. The quality of this cream is good, and for the price, it 

## Evaluation function 
- Evaluation using Review Text Quality (ROUGE /BERTScore) & Rating Accuracy (RMSE)

In [45]:
!pip install rouge-score
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [47]:
rich_df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq,word_count
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,"Flawless Finish Foundation, Colour Changing Fo...",,,NaN,CIDBEST,All Beauty,1,158
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,"Flawless Liquid Foundation Cream, Liquid Found...",,,NaN,Cherioll,All Beauty,2,97
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,"Rapid Reduction Eye Cream, Under Eye Cream, Un...",,,NaN,CIDBEST,All Beauty,3,137
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,"Concealer Cream,Makeup concealer,Skin Lighteni...",,,NaN,SCOBUTY,All Beauty,4,132
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,W-Airfit Primer Face Makeup Base Pink Isolatio...,,,NaN,Lofu,All Beauty,5,124


In [49]:

from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
import json
import re

# ── INSTALL FIRST IF NEEDED ──────────────────────────────────────
# pip install rouge-score
# pip install bert-score

# ── STEP 1: ROUGE SCORER ────────────────────────────────────────

def compute_rouge(generated: str, reference: str) -> dict:
    """
    Computes ROUGE-1, ROUGE-2, ROUGE-L between generated and reference review.
    Returns F1 scores for each.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    scores = scorer.score(reference, generated)

    return {
        'rouge1' : round(scores['rouge1'].fmeasure,  4),
        'rouge2' : round(scores['rouge2'].fmeasure,  4),
        'rougeL' : round(scores['rougeL'].fmeasure,  4),
    }


# ── STEP 2: BERTSCORE ────────────────────────────────────────────

def compute_bertscore(generated_list: list, reference_list: list) -> list:
    """
    Computes BERTScore for a batch of generated vs reference review pairs.
    Runs in one batch — much faster than calling per review.
    Returns a list of F1 scores, one per pair.
    """
    P, R, F1 = bert_score(
        generated_list,
        reference_list,
        lang        = "en",
        model_type  = "roberta-large", #"distilbert-base-uncased",  # lighter model — faster
        verbose     = False
    )
    return [round(f.item(), 4) for f in F1]


# ── STEP 3: RMSE ─────────────────────────────────────────────────

def compute_rmse(true_ratings: list, pred_ratings: list) -> float:
    true = np.array(true_ratings, dtype=float)
    pred = np.array(pred_ratings, dtype=float)
    return round(float(np.sqrt(np.mean((true - pred) ** 2))), 4)


# ── STEP 4: PARSE SIMULATED OUTPUT ──────────────────────────────

def parse_simulated_output(raw: str, fallback_rating: float = 3.0) -> dict:
    """
    Parses the structured output from simulate_review.
    Handles messy LLM outputs gracefully.
    """
    result = {
        'reasoning' : None,
        'rating'    : None,
        'title'     : None,
        'review'    : None
    }

    for field in ['REASONING', 'RATING', 'TITLE', 'REVIEW']:
        pattern = rf"{field}:\s*(.*?)(?=\n[A-Z]{{2,}}:|$)"
        match   = re.search(pattern, raw, re.DOTALL)
        if match:
            result[field.lower()] = match.group(1).strip()

    # safely cast rating to int
    try:
        raw_rating      = str(result['rating']).strip()
        result['rating'] = int(re.search(r'\d', raw_rating).group())
        result['rating'] = max(1, min(5, result['rating']))  # clamp to 1-5
    except (TypeError, AttributeError, ValueError):
        result['rating'] = round(fallback_rating)

    return result


# ── STEP 5: FULL EVALUATION PIPELINE ────────────────────────────

def evaluate_pipeline(
    persona_df,
    clean_df,
    model,
    n_users         = 20,
    nigerian_mode   = False,
    verbose         = True
):
    """
    Full evaluation pipeline — RMSE + ROUGE + BERTScore.

    For each user:
    - Holds out their last review as ground truth
    - Simulates their review for that item using their history
    - Scores the simulation against ground truth

    Parameters:
        persona_df    : output of extract_statistical_persona()
        clean_df      : your cleaned amazon dataframe
        model         : Gemini model instance
        n_users       : how many users to evaluate on
        nigerian_mode : whether to apply Nigerian conditioning
        verbose       : print per-user results as they run

    Returns:
        results_df    : full results dataframe
        summary       : dict of aggregate metrics
    """

    results = []
    skipped = 0

    print("=" * 60)
    print("EVALUATION PIPELINE — ROUGE + BERTScore + RMSE")
    print("=" * 60)
    print(f"Users to evaluate : {n_users}")
    print(f"Nigerian mode     : {nigerian_mode}")
    print()

    for idx, (_, user_row) in enumerate(persona_df.head(n_users).iterrows()):

        user_id      = user_row['user_id']
        user_reviews = clean_df[clean_df['user_id'] == user_id].sort_values('timestamp')

        # need at least 5 reviews — 4 for history, 1 for holdout
        if len(user_reviews) < 5:
            skipped += 1
            continue

        # ── HOLDOUT SPLIT ────────────────────────────────────────
        holdout = user_reviews.iloc[-1]   # last review = ground truth
        history = user_reviews.iloc[:-1]  # everything before = persona input

        # rebuild sample_reviews from history only — never include holdout
        user_row_temp                   = user_row.copy()
        user_row_temp['sample_reviews'] = history['text'].tolist()[-5:]

        # ── ITEM DETAILS ─────────────────────────────────────────
        item_asin  = holdout.get('asin', 'UNKNOWN')
        item_title = holdout.get('product_title', 'unknown')

        # IMPORTANT: never pass holdout review text as description
        # use only product metadata — title is enough for prototype
        item_description = f"Amazon product: {item_title}"

        # ── SIMULATE ─────────────────────────────────────────────
        try:
            prompt = build_user_prompt(
                user_row        = user_row_temp,
                item_asin       = item_asin,
                item_title      = item_title,
                item_description= item_description,
                nigerian_mode   = nigerian_mode
            )
            response = model.generate_content(prompt)
            parsed   = parse_simulated_output(
                raw             = response.text.strip(),
                fallback_rating = user_row['avg_rating']
            )
        except Exception as e:
            print(f"  ⚠️  User {user_id} — generation failed: {e}")
            skipped += 1
            continue

        # skip if review text is empty
        if not parsed['review']:
            skipped += 1
            continue

        # ── ROUGE ─────────────────────────────────────────────────
        rouge_scores = compute_rouge(
            generated = parsed['review'],
            reference = str(holdout['text'])
        )

        results.append({
            'user_id'          : user_id,
            'true_rating'      : int(holdout['rating']),
            'pred_rating'      : parsed['rating'],
            'true_review'      : str(holdout['text']),
            'generated_review' : parsed['review'],
            'reasoning'        : parsed['reasoning'],
            'rouge1'           : rouge_scores['rouge1'],
            'rouge2'           : rouge_scores['rouge2'],
            'rougeL'           : rouge_scores['rougeL'],
            'bertscore'        : None,   # filled in batch below
        })

        if verbose:
            print(f"[{idx+1}/{n_users}] User {user_id[:12]}... "
                  f"| True: {int(holdout['rating'])}★ "
                  f"| Pred: {parsed['rating']}★ "
                  f"| ROUGE-1: {rouge_scores['rouge1']:.3f}")

    # ── BERTSCORE IN BATCH ────────────────────────────────────────
    # run all at once — much faster than per-review calls
    if results:
        print(f"\nRunning BERTScore on {len(results)} pairs...")
        bert_scores = compute_bertscore(
            generated_list = [r['generated_review'] for r in results],
            reference_list = [r['true_review']       for r in results]
        )
        for i, score in enumerate(bert_scores):
            results[i]['bertscore'] = score

    # ── BUILD RESULTS DATAFRAME ───────────────────────────────────
    results_df = pd.DataFrame(results)

    # ── AGGREGATE SUMMARY ─────────────────────────────────────────
    summary = {}
    if len(results_df) > 0:
        summary = {
            'n_evaluated'    : len(results_df),
            'n_skipped'      : skipped,
            'rmse'           : compute_rmse(
                                    results_df['true_rating'].tolist(),
                                    results_df['pred_rating'].tolist()
                               ),
            'avg_rouge1'     : round(results_df['rouge1'].mean(),     4),
            'avg_rouge2'     : round(results_df['rouge2'].mean(),     4),
            'avg_rougeL'     : round(results_df['rougeL'].mean(),     4),
            'avg_bertscore'  : round(results_df['bertscore'].mean(),  4),
        }

        print(f"\n{'=' * 60}")
        print(f"EVALUATION RESULTS")
        print(f"{'=' * 60}")
        print(f"  Users evaluated  : {summary['n_evaluated']}")
        print(f"  Users skipped    : {summary['n_skipped']}")
        print(f"\n  Rating RMSE      : {summary['rmse']}")
        print(f"  (target: < 1.0  | strong: < 0.8)")
        print(f"\n  Avg ROUGE-1      : {summary['avg_rouge1']}")
        print(f"  Avg ROUGE-2      : {summary['avg_rouge2']}")
        print(f"  Avg ROUGE-L      : {summary['avg_rougeL']}")
        print(f"  (ROUGE-1 > 0.3 is reasonable for open-ended generation)")
        print(f"\n  Avg BERTScore    : {summary['avg_bertscore']}")
        print(f"  (BERTScore > 0.85 is strong for this task)")
        print(f"{'=' * 60}\n")

    return results_df, summary


# ── RUN IT ───────────────────────────────────────────────────────

results_df, summary = evaluate_pipeline(
    persona_df    = persona_df_claude,
    clean_df      = rich_df,
    model         = model,
    n_users       = 20,
    nigerian_mode = False,
    verbose       = True
)

# inspect the worst performing simulations
# these tell you where your persona extraction is weakest
worst = results_df.nsmallest(3, 'bertscore')
for _, row in worst.iterrows():
    print(f"\nUser: {row['user_id']}")
    print(f"True   : {row['true_review'][:200]}")
    print(f"Generated: {row['generated_review'][:200]}")
    print(f"BERTScore: {row['bertscore']}")

EVALUATION PIPELINE — ROUGE + BERTScore + RMSE
Users to evaluate : 20
Nigerian mode     : False

[1/20] User AE7P3G7DWP3V... | True: 5★ | Pred: 5★ | ROUGE-1: 0.383
[2/20] User AEYKTZXAWOPJ... | True: 5★ | Pred: 5★ | ROUGE-1: 0.467
[3/20] User AFR4BTNWATNG... | True: 5★ | Pred: 5★ | ROUGE-1: 0.344
[4/20] User AFSCJNRG4BAG... | True: 5★ | Pred: 5★ | ROUGE-1: 0.319
[5/20] User AFUBMCVI5J6G... | True: 4★ | Pred: 3★ | ROUGE-1: 0.328
[6/20] User AGFN3252BBTY... | True: 5★ | Pred: 5★ | ROUGE-1: 0.375
[7/20] User AGUTZC4GHLTG... | True: 5★ | Pred: 5★ | ROUGE-1: 0.428

Running BERTScore on 7 pairs...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



EVALUATION RESULTS
  Users evaluated  : 7
  Users skipped    : 0

  Rating RMSE      : 0.378
  (target: < 1.0  | strong: < 0.8)

  Avg ROUGE-1      : 0.3775
  Avg ROUGE-2      : 0.0605
  Avg ROUGE-L      : 0.2045
  (ROUGE-1 > 0.3 is reasonable for open-ended generation)

  Avg BERTScore    : 0.8581
  (BERTScore > 0.85 is strong for this task)


User: AFUBMCVI5J6G4F2RFQGTDRXLE6VQ
True   : I found this doesn't fit very comfortable on my face and doesn't quite lay on my chin area the way I had hoped. I find I have to turn it nearly all the way up to feel some pulses . There is no plate d
Generated: So, fantastic shipping on this item. It came very fast, which was great. The packaging was just okay. It was in a standard box, nothing really special about it. The V-face massager itself feels a bit 
BERTScore: 0.8323

User: AGFN3252BBTYUJUUDQMAGYZNUT5A_2_1
True   : [[VIDEOID:e0db87669578d949a427fad3a534e022]] The wig is made of soft hair material and transplanted from natural hair. With wavy

## gaps i see 
maybe i need to input an ai to extract th details about users so that the profile building on the users will be better this one here is shit 
so the persona will have an ai (maybe gemini 2.5 ) that will extract details from reviews and then another ai will use the extracted details to build a sweet review taking every thing the user likes and talks about into consideration 